# Part 4: Up to **Four** Previous Characters: a Comparison

* **Part 1:** 1 previous character
* **Part 2:** 2 previous characters
* **Part 3:** 3 previous characters
* **Part 4 (this notebook):** we add a model that sees **4** previous characters and compare all four on the same text.

| context | what the model can "know" |
|---|---|
| `␣` → ? | any word can start here |
| `e␣` → ? | probably the word after *the* |
| `he␣` → ? | still: the word after *the* (cat, dog, mat, log) |
| `the␣` → ? | the same. **4 characters do not help here** |
| `␣on␣` → ? | after *on* always comes *the* → `t` |
| `and␣` → ? | after *and* always comes *the* → `t` |

### The input layer for 4 characters

Four one-hot vectors are placed side by side, so the input size is $4V$:

$$
x = [\underbrace{\text{one-hot}(c_{t-4})}_{V},\ \underbrace{\text{one-hot}(c_{t-3})}_{V},\ \underbrace{\text{one-hot}(c_{t-2})}_{V},\ \underbrace{\text{one-hot}(c_{t-1})}_{V}]
$$

The hidden layer, softmax, loss and backpropagation are the same as in Parts 1-3.

**Question to keep in mind:** Part 3 showed that 3 characters already started to memorize. Will 4 characters still help?

## 1. Helper functions

In [ ]:
import math
import random
import matplotlib.pyplot as plt
from collections import Counter


def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def sigmoid_derivative(h):
    return h * (1.0 - h)


def softmax(scores):
    m = max(scores)
    exps = [math.exp(s - m) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]


def one_hot(index, size):
    vec = [0.0] * size
    vec[index] = 1.0
    return vec


def show(ch):
    return {" ": "␣", "\n": "↵"}.get(ch, ch)


def show_ctx(context):
    return "".join(show(ch) for ch in context)

## 2. Training data (same text as in Parts 1 and 2)

In [ ]:
text = (
    "the cat sat on the mat. "
    "the dog sat on the log. "
    "the cat and the dog ran to the mat. "
    "the dog had a hat and the cat had a bat."
)

vocab = sorted(set(text))
V = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}

print(text)
print("\nVocabulary:", [show(c) for c in vocab], " V =", V)

## 3. Encoding a context of several characters

`encode("cat.")` gives a vector of length $4V$: one-hot of `c`, `a`, `t` and `.`.

In [ ]:
def encode(context):
    x = []
    for ch in context:                 # glue the one-hot vectors together
        x += one_hot(char_to_idx[ch], V)
    return x


def make_pairs(context_size):
    # (context string, index of the next character)
    return [(text[t - context_size:t], char_to_idx[text[t]])
            for t in range(context_size, len(text))]


x = encode("cat.")
print("encode('cat.') has length", len(x), "= 4 x", V)
for part, ch in enumerate("cat."):
    print(f"   part {part + 1} ({ch}):", x[part * V:(part + 1) * V])

print("\nFirst 8 training examples with context_size = 4:")
for ctx, nxt in make_pairs(4)[:8]:
    print(f"   '{show_ctx(ctx)}'  ->  '{show(idx_to_char[nxt])}'")

## 4. The network: initialization, forward pass, backward pass

Same formulas as in Part 1. The only difference is the input size `n_in = context_size * V`.

* $\delta^o_k = p_k - y_k$
* $\delta^h_j = \left(\sum_k \delta^o_k U_{kj}\right) h_j(1-h_j)$
* $\partial \ell / \partial U_{kj} = \delta^o_k h_j$, $\quad \partial \ell / \partial W_{ji} = \delta^h_j x_i$

In [ ]:
def init_model(context_size, H, seed=0):
    rnd = random.Random(seed)
    n_in = context_size * V
    return {
        "context_size": context_size,
        "H": H,
        "W": [[rnd.uniform(-0.5, 0.5) for _ in range(n_in)] for _ in range(H)],
        "b": [0.0] * H,
        "U": [[rnd.uniform(-0.5, 0.5) for _ in range(H)] for _ in range(V)],
        "c": [0.0] * V,
    }


def forward(model, context):
    W, b, U, c, H = model["W"], model["b"], model["U"], model["c"], model["H"]
    x = encode(context)
    n_in = len(x)

    z = [sum(W[j][i] * x[i] for i in range(n_in)) + b[j] for j in range(H)]
    h = [sigmoid(zj) for zj in z]

    s = [sum(U[k][j] * h[j] for j in range(H)) + c[k] for k in range(V)]
    p = softmax(s)
    return x, h, p


def train_step(model, context, next_idx, learning_rate):
    W, b, U, c, H = model["W"], model["b"], model["U"], model["c"], model["H"]

    # --- FORWARD PASS ---
    x, h, p = forward(model, context)
    n_in = len(x)
    loss = -math.log(p[next_idx])

    # --- BACKWARD PASS ---
    y = one_hot(next_idx, V)
    delta_o = [p[k] - y[k] for k in range(V)]
    delta_h = [
        sum(delta_o[k] * U[k][j] for k in range(V)) * sigmoid_derivative(h[j])
        for j in range(H)
    ]

    # --- PARAMETER UPDATE ---
    for k in range(V):
        for j in range(H):
            U[k][j] -= learning_rate * delta_o[k] * h[j]
        c[k] -= learning_rate * delta_o[k]

    for j in range(H):
        for i in range(n_in):
            if x[i] != 0.0:            # x is one-hot: other gradients are 0
                W[j][i] -= learning_rate * delta_h[j] * x[i]
        b[j] -= learning_rate * delta_h[j]

    return loss


def train(context_size, H=16, epochs=400, learning_rate=0.3, seed=0):
    model = init_model(context_size, H, seed)
    pairs = make_pairs(context_size)
    rnd = random.Random(seed)
    history = []
    for epoch in range(1, epochs + 1):
        rnd.shuffle(pairs)
        total = sum(train_step(model, ctx, nxt, learning_rate) for ctx, nxt in pairs)
        history.append(total / len(pairs))
        if epoch == 1 or epoch % 100 == 0:
            print(f"   epoch {epoch:<4} loss {history[-1]:.4f}")
    return model, history

## 5. Train all four models

| model | context | input size |
|---|---|---|
| A | 1 character | V |
| B | 2 characters | 2V |
| C | 3 characters | 3V |
| D | 4 characters | 4V |

All four use the same hidden size, learning rate, number of epochs and random seed, so the comparison is fair. Training takes around 20 seconds.

In [ ]:
names = {1: "A: 1 char", 2: "B: 2 chars", 3: "C: 3 chars", 4: "D: 4 chars"}
colors = {1: "royalblue", 2: "darkorange", 3: "crimson", 4: "purple"}

models, histories = {}, {}
for n in [1, 2, 3, 4]:
    print(f"Model {names[n]}   (input size {n * V})")
    models[n], histories[n] = train(context_size=n)
    print()

plt.figure(figsize=(8, 4))
for n in [1, 2, 3, 4]:
    plt.plot(histories[n], color=colors[n], linewidth=2, label=names[n])
plt.axhline(math.log(V), color="gray", linestyle="--", label="random guessing")
plt.title("Learning curves")
plt.xlabel("Epoch")
plt.ylabel("Average cross-entropy loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 6. 🎯 Predict the next character, starting from a position in the text

At every position each model gets the **real** previous characters from the text (1, 2 or 3 of them) and guesses the next character.

✏️ Change `start_position` and `num_steps`.

In [ ]:
start_position = 24
num_steps = 23


def predict(model, pos):
    # predict text[pos + 1] from the real characters that end at position pos
    n = model["context_size"]
    context = text[pos + 1 - n:pos + 1]
    _, _, p = forward(model, context)
    best = max(range(V), key=lambda k: p[k])
    return context, idx_to_char[best], p[best]


print(text)
print(" " * start_position + "^ start here\n")

header = f"{'pos':<4}| {'actual':<6}"
for n in [1, 2, 3, 4]:
    header += f"| {names[n]:<18}"
print(header)
print(f"{'':<4}| {'':<6}" + f"| {'ctx   guess prob':<18}" * 4)
print("-" * len(header))

start = max(start_position, 3)              # need at least 4 characters before
end = min(start + num_steps, len(text) - 1)
correct = {1: 0, 2: 0, 3: 0, 4: 0}
guesses = {1: "", 2: "", 3: "", 4: ""}
for pos in range(start, end):
    actual = text[pos + 1]
    row = f"{pos:<4}| '{show(actual)}'   "
    for n in [1, 2, 3, 4]:
        ctx, g, pr = predict(models[n], pos)
        ok = g == actual
        correct[n] += ok
        guesses[n] += g
        row += f"| {repr(show_ctx(ctx)):<5} '{show(g)}' {pr:.2f} {'✓' if ok else '✗'} "
    print(row)

steps = end - start
print()
print(f"{'Real text':<12}:", repr(text[start + 1:end + 1]))
for n in [1, 2, 3, 4]:
    print(f"{names[n]:<12}:", repr(guesses[n]),
          f"  accuracy {correct[n]}/{steps} = {100 * correct[n] / steps:.0f}%")

## 7. Accuracy and loss over the whole text

Every model is tested on the same positions, from position 4 to the end of the text.

In [ ]:
def evaluate(model):
    n = model["context_size"]
    correct = 0
    total_loss = 0.0
    positions = range(4, len(text))          # same positions for all models
    for t in positions:
        _, _, p = forward(model, text[t - n:t])
        target = char_to_idx[text[t]]
        correct += max(range(V), key=lambda k: p[k]) == target
        total_loss += -math.log(p[target])
    return correct / len(positions), total_loss / len(positions)


results = {n: evaluate(models[n]) for n in [1, 2, 3, 4]}

print(f"{'model':<12} | {'accuracy':<8} | {'loss':<6} | {'parameters':<10}")
print("-" * 46)
for n in [1, 2, 3, 4]:
    H = models[n]["H"]
    params = H * n * V + H + V * H + V
    acc, loss = results[n]
    print(f"{names[n]:<12} | {100 * acc:6.1f}%  | {loss:.3f}  | {params:<10}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
labels = [names[n] for n in [1, 2, 3, 4]]
cols = [colors[n] for n in [1, 2, 3, 4]]

accs = [100 * results[n][0] for n in [1, 2, 3, 4]]
bars = ax1.bar(labels, accs, color=cols)
for bar, a in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width() / 2, a + 1, f"{a:.0f}%", ha="center")
ax1.set_ylim(0, 100)
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Next-character accuracy")

losses = [results[n][1] for n in [1, 2, 3, 4]]
bars = ax2.bar(labels, losses, color=cols)
for bar, l in zip(bars, losses):
    ax2.text(bar.get_x() + bar.get_width() / 2, l + 0.03, f"{l:.2f}", ha="center")
ax2.set_ylabel("Average cross-entropy loss")
ax2.set_title("Loss (lower is better)")

plt.tight_layout()
plt.show()

## 8. Where does more context help?

Compare the probability distributions of all four models for one position. ✏️ Try positions right after a space, for example 27 (after `the␣`) or 37 (after `␣on␣`).

In [ ]:
position = 37   # predict text[position + 1]

actual = text[position + 1]
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=True)
for ax, n in zip(axes.flat, [1, 2, 3, 4]):
    ctx = text[position + 1 - n:position + 1]
    _, _, p = forward(models[n], ctx)
    bar_colors = ["seagreen" if ch == actual else "lightsteelblue" for ch in vocab]
    ax.bar([show(ch) for ch in vocab], p, color=bar_colors)
    ax.set_title(f"{names[n]}: context '{show_ctx(ctx)}'")
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", linestyle="--", alpha=0.6)
axes[0][0].set_ylabel("Probability")
axes[1][0].set_ylabel("Probability")
fig.suptitle(f"Predicting position {position + 1}: real next character is '{show(actual)}' (green)")
plt.tight_layout()
plt.show()

Some contexts where the extra characters matter:

In [ ]:
for ctx4 in [" on ", "the ", "and ", "had ", "sat ", " ran", "d a "]:
    row = ""
    for n in [1, 2, 3, 4]:
        ctx = ctx4[-n:]
        _, _, p = forward(models[n], ctx)
        k = max(range(V), key=lambda k: p[k])
        row += f" {repr(show_ctx(ctx)):>6} -> '{show(idx_to_char[k])}' {p[k]:.2f} |"
    print(row)

## 9. Let all four models write on their own

Each model starts with the first 4 characters of the text (`the␣`) and then feeds its own predictions back in as input.

In [ ]:
def generate(model, length=60, greedy=False, seed=None):
    rnd = random.Random(seed)
    n = model["context_size"]
    out = text[:4]                       # same start for all models
    for _ in range(length):
        _, _, p = forward(model, out[-n:])
        if greedy:
            k = max(range(V), key=lambda k: p[k])
        else:
            k = rnd.choices(range(V), weights=p)[0]
        out += idx_to_char[k]
    return out


for n in [1, 2, 3, 4]:
    print(f"{names[n]:<11} greedy :", repr(generate(models[n], greedy=True)))
    for s in range(2):
        print(f"{names[n]:<11} sample :", repr(generate(models[n], seed=s)))
    print()

## 10. The theoretical limit, and the danger of memorizing

The lowest loss a model can reach depends on the context length. We also count how many **different contexts** appear in the text and how many of them appear **only once**.

In [ ]:
def best_possible_loss(n):
    ctx_next = Counter((text[t - n:t], text[t]) for t in range(n, len(text)))
    ctx = Counter(text[t - n:t] for t in range(n, len(text)))
    total = len(text) - n
    return -sum(cnt * math.log(cnt / ctx[c]) for (c, _), cnt in ctx_next.items()) / total


print(f"{'context':<8} | {'best loss':<9} | {'our loss':<8} | {'distinct contexts':<17} | {'seen only once':<14} | {'possible contexts':<17}")
print("-" * 88)
for n in [1, 2, 3, 4, 5, 6]:
    ctx = Counter(text[t - n:t] for t in range(n, len(text)))
    once = sum(1 for c in ctx.values() if c == 1)
    ours = f"{histories[n][-1]:.4f}" if n in histories else "-"
    print(f"{n:<8} | {best_possible_loss(n):<9.4f} | {ours:<8} | {len(ctx):<17} | {once:<14} | {V ** n:<17}")

**How to read this table:**

* With more context, the **best possible loss goes down**, because the model can tell more situations apart.
* But the number of **possible contexts grows as $V^n$**, while our text stays the same length. With 4 characters most possible combinations never appear at all, and many of those that do appear are **seen only once**.
* A context seen only once is easy to **memorize**, but the model learns nothing general from it. On new text, a longer-context model can therefore do **worse** than a shorter one. This is called **overfitting**, and it is why more context is not automatically better unless you also have more data.

## 11. Test on text the models have never seen

The text below uses the same characters, but sentences the models were not trained on.

✏️ Write your own test sentence (use only characters from the vocabulary).

In [ ]:
test_text = "the cat ran to the log. the dog sat on the hat."

missing = set(test_text) - set(vocab)
assert not missing, f"These characters are not in the vocabulary: {missing}"

print(f"{'model':<12} | {'accuracy on new text':<20} | {'loss on new text':<16}")
print("-" * 54)
for n in [1, 2, 3, 4]:
    correct = 0
    loss = 0.0
    positions = range(4, len(test_text))
    for t in positions:
        _, _, p = forward(models[n], test_text[t - n:t])
        target = char_to_idx[test_text[t]]
        correct += max(range(V), key=lambda k: p[k]) == target
        loss += -math.log(p[target])
    print(f"{names[n]:<12} | {100 * correct / len(positions):18.1f}%  | {loss / len(positions):.3f}")

## 12. Summary and exercises

**What we saw:**
1. 1 → 2 characters: a **big** improvement.
2. 2 → 3 characters: a smaller improvement.
3. 3 → 4 characters: a **smaller** improvement again on the training text (about 88% → 91%). It also helped on our new test sentence, but only because that sentence reuses whole phrases from the training text (*sat on the*, *the dog*). Half of the 4-character contexts are seen only once, so on really different text the 4-character model mostly has memorized patterns to rely on.
4. Each extra character adds $H \times V$ weights, and the number of possible contexts grows as $V^n$ (65,536 for 4 characters here), while our text has only about 120 positions.

**The lesson:** a longer context is only useful if there is enough data to learn from it. This is the main limitation of fixed-window models, and it led to:
* **Character embeddings** (Bengio et al., 2003): each character becomes a small learned vector, so similar contexts can share what they learn.
* **Recurrent networks (RNN, LSTM):** they read the text one character at a time and keep a memory, so the context has no fixed length.
* **Transformers:** they look at all previous characters at once using attention.

### Exercises
1. Make the training text 3-4 times longer (more sentences of the same style). Does the 4-character model start to beat the 3-character model on new text?
2. Find positions in section 8 where the 4-character model is right and the 3-character model is wrong. Which word does the 4th character "reveal"?
3. Try `H = 4` for all models. Which model suffers most from having too few hidden neurons?
4. Plot accuracy on the training text and on the new text against context size (1, 2, 3, 4) on one chart. Where do the two curves separate?